In [4]:
import json

with open("/Users/Mourya/projects/aircraft-conflict-gnn/processed_data/gcn_results.json") as f:
    gcn = json.load(f)

print(gcn)

{'model': 'GCN', 'precision': 0.004513540621865597, 'recall': 0.9, 'f1': 0.008982035928143712, 'roc_auc': 0.9804823907877342}


In [5]:
import os
import time

path = "/Users/Mourya/projects/aircraft-conflict-gnn/processed_data/gcn_results.json"

print("Modified:", time.ctime(os.path.getmtime(path)))

Modified: Sat Aug  1 01:52:25 2026


In [7]:
import json

path = "/Users/Mourya/projects/aircraft-conflict-gnn/processed_data/gcn_results.json"

with open(path) as f:
    gcn = json.load(f)

print(gcn)

{'model': 'GCN', 'precision': 0.004513540621865597, 'recall': 0.9, 'f1': 0.008982035928143712, 'roc_auc': 0.9804823907877342}


In [9]:
import pandas as pd
import json

with open("/Users/Mourya/projects/aircraft-conflict-gnn/processed_data/gcn_results.json") as f:
    gcn = json.load(f)

comparison = pd.DataFrame([
    {"Model": "Rule-based", "Precision": 0.0555, "Recall": 0.3402, "F1": 0.0955, "ROC_AUC": None},
    {"Model": "XGBoost", "Precision": 0.6119, "Recall": 0.8367, "F1": 0.7069, "ROC_AUC": 0.9998},
    {"Model": "GCN", "Precision": gcn["precision"], "Recall": gcn["recall"], "F1": gcn["f1"], "ROC_AUC": gcn["roc_auc"]},
    {"Model": "GAT", "Precision": 0.0048, "Recall": 0.3333, "F1": 0.0094, "ROC_AUC": 0.9420},
])
comparison.to_csv("processed_data/model_comparison.csv", index=False)
print(comparison)

        Model  Precision  Recall        F1   ROC_AUC
0  Rule-based   0.055500  0.3402  0.095500       NaN
1     XGBoost   0.611900  0.8367  0.706900  0.999800
2         GCN   0.004514  0.9000  0.008982  0.980482
3         GAT   0.004800  0.3333  0.009400  0.942000


In [10]:
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class ConflictGAT(nn.Module):
    def __init__(self, node_dim=6, edge_dim=5, hidden_dim=64, heads=2):
        super().__init__()
        self.gat1 = GATConv(node_dim, hidden_dim, heads=heads, concat=True, edge_dim=edge_dim)
        self.gat2 = GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, edge_dim=edge_dim)
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_dim, 64), nn.ReLU(), nn.Linear(64, 1)
        )
    def forward(self, data):
        x = F.elu(self.gat1(data.x, data.edge_index, data.edge_attr))
        x = self.gat2(x, data.edge_index, data.edge_attr)
        src, dst = x[data.edge_index[0]], x[data.edge_index[1]]
        return self.edge_mlp(torch.cat([src, dst, data.edge_attr], dim=1)).squeeze(-1)

device = torch.device("cpu")
model = ConflictGAT().to(device)
model.load_state_dict(torch.load("models/conflict_gat_large_best.pth", map_location=device, weights_only=False))
model.eval()

test_graphs = torch.load("processed_data/test_graphs.pt", weights_only=False)

# find a real true positive (high-confidence correct catch) for the case study
target_graph = next(g for g in test_graphs if g.y.sum() > 0)

with torch.no_grad():
    _, (edge_idx_out, attn_weights) = model.gat1(
        target_graph.x, target_graph.edge_index, target_graph.edge_attr,
        return_attention_weights=True
    )

pos_idx = (target_graph.y == 1).nonzero(as_tuple=True)[0]
i = pos_idx[0].item()
print("Case study edge features (h_dist, v_dist, closing_rate, bearing_diff, time_to_cpa):")
print(target_graph.edge_attr[i].tolist())
print("Attention weight(s) across heads:", attn_weights[i].tolist())

/Users/Mourya/miniforge3/envs/aircraft-conflict-gnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Case study edge features (h_dist, v_dist, closing_rate, bearing_diff, time_to_cpa):
[21.529813766479492, 1000.0000610351562, 0.13295966386795044, 72.79090118408203, 161.9274139404297]
Attention weight(s) across heads: [0.0, 0.0]


In [11]:
state = torch.load("models/conflict_gat_large_best.pth", map_location="cpu", weights_only=False)
for k, v in state.items():
    print(k, tuple(v.shape))

gat1.att_src (1, 2, 64)
gat1.att_dst (1, 2, 64)
gat1.att_edge (1, 2, 64)
gat1.bias (128,)
gat1.lin.weight (128, 6)
gat1.lin_edge.weight (128, 5)
gat2.att_src (1, 1, 64)
gat2.att_dst (1, 1, 64)
gat2.att_edge (1, 1, 64)
gat2.bias (64,)
gat2.lin.weight (64, 128)
gat2.lin_edge.weight (64, 5)
edge_mlp.0.weight (64, 133)
edge_mlp.0.bias (64,)
edge_mlp.2.weight (1, 64)
edge_mlp.2.bias (1,)


In [12]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,5))
plt.hist(attn_weights.mean(dim=1).detach().numpy(), bins=30)
plt.axvline(attn_weights.mean(dim=1)[i].item(), color="red", linestyle="--", label="Case study edge (true positive)")
plt.xlabel("Mean attention weight")
plt.title("GAT Attention Distribution — True Positive Case Study")
plt.legend()
plt.savefig("results/gat_attention_case_study.png", bbox_inches="tight")
plt.close()

In [13]:
attn_mean = attn_weights.mean(dim=1)
pos_idx = (target_graph.y == 1).nonzero(as_tuple=True)[0]
pos_attn = attn_mean[pos_idx]
best_i = pos_idx[pos_attn.argmax()].item()
print("Best-attended true positive edge features:", target_graph.edge_attr[best_i].tolist())
print("Attention weight:", attn_mean[best_i].item())

Best-attended true positive edge features: [21.529813766479492, 1000.0000610351562, 0.13295966386795044, 72.79090118408203, 161.9274139404297]
Attention weight: 0.0


In [14]:
# get the real src/dst for your case-study edge from the ORIGINAL edge_index
pos_idx = (target_graph.y == 1).nonzero(as_tuple=True)[0]
orig_i = pos_idx[0].item()
src_node = target_graph.edge_index[0, orig_i].item()
dst_node = target_graph.edge_index[1, orig_i].item()
print(f"Looking for edge: {src_node} -> {dst_node}")

# search for this exact (src, dst) pair in the returned edge_idx_out
match = ((edge_idx_out[0] == src_node) & (edge_idx_out[1] == dst_node)).nonzero(as_tuple=True)[0]

if len(match) > 0:
    correct_i = match[0].item()
    print("Correct attention weight:", attn_weights[correct_i].tolist())
else:
    print("Edge not found in returned index — checking self-loop count")
    print("Original edges:", target_graph.edge_index.shape[1])
    print("Returned edges (with self-loops):", edge_idx_out.shape[1])

Looking for edge: 1 -> 21
Correct attention weight: [0.0, 0.0]


In [15]:
from torch_geometric.nn import GATConv

# temporarily swap in a no-self-loop version for clean attention extraction
inspect_layer = GATConv(6, 64, heads=2, concat=True, edge_dim=5, add_self_loops=False)
inspect_layer.load_state_dict(model.gat1.state_dict())
inspect_layer.eval()

with torch.no_grad():
    _, (edge_idx_out2, attn_weights2) = inspect_layer(
        target_graph.x, target_graph.edge_index, target_graph.edge_attr,
        return_attention_weights=True
    )

print("Edge count now matches original:", edge_idx_out2.shape[1] == target_graph.edge_index.shape[1])
print("Attention weight for case-study edge:", attn_weights2[orig_i].tolist())

Edge count now matches original: True
Attention weight for case-study edge: [0.0, 0.0]


In [16]:
dst_node = target_graph.edge_index[1, orig_i].item()
incoming_mask = (edge_idx_out2[1] == dst_node)
incoming_attn = attn_weights2[incoming_mask].mean(dim=1)
print(f"Node {dst_node} has {incoming_mask.sum().item()} incoming edges")
print("Attention weight distribution at this node:")
print("  max:", incoming_attn.max().item())
print("  top 5:", incoming_attn.topk(min(5, len(incoming_attn))).values.tolist())
print("  how many are effectively zero (<1e-6):", (incoming_attn < 1e-6).sum().item())

Node 21 has 21 incoming edges
Attention weight distribution at this node:
  max: 1.0
  top 5: [1.0, 0.0, 0.0, 0.0, 0.0]
  how many are effectively zero (<1e-6): 20


In [19]:
import torch
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# reload both models with their final trained weights
val_graphs = torch.load("processed_data/val_graphs.pt", weights_only=False)
test_graphs = torch.load("processed_data/test_graphs.pt", weights_only=False)

from torch_geometric.loader import DataLoader
val_loader = DataLoader(val_graphs, batch_size=8, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=8, shuffle=False)

def get_probs_labels(model, loader, device):
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch)
            p = torch.sigmoid(logits)
            probs.extend(p.cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
    return np.array(probs), np.array(labels)

device = torch.device("cpu")

# --- GAT threshold tuning (validation only) ---
val_probs, val_labels = get_probs_labels(model, val_loader, device)  # your loaded GAT

thresholds = np.arange(0.01, 1.00, 0.01)
best_t, best_f1 = 0.5, -1
for t in thresholds:
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(val_labels, preds, zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"GAT — best threshold from validation: {best_t:.2f} (val F1={best_f1:.4f})")

# --- apply frozen threshold to test, ONCE ---
test_probs, test_labels = get_probs_labels(model, test_loader, device)
test_preds = (test_probs >= best_t).astype(int)

print("\nGAT final test results at frozen threshold:")
print("Precision:", precision_score(test_labels, test_preds, zero_division=0))
print("Recall:", recall_score(test_labels, test_preds, zero_division=0))
print("F1:", f1_score(test_labels, test_preds, zero_division=0))
print(confusion_matrix(test_labels, test_preds))

GAT — best threshold from validation: 0.98 (val F1=0.0853)

GAT final test results at frozen threshold:
Precision: 0.0030120481927710845
Recall: 0.06666666666666667
F1: 0.005763688760806916
[[130207    662]
 [    28      2]]


In [21]:
region_final["climb_rate_fpm"] = (region_final["dalt"] * 3.28084) / region_final["dt"] * 60
MAX_CLIMB_RATE = 6000  # now correctly ft/min
region_final = region_final[
    (region_final["climb_rate_fpm"].abs() <= MAX_CLIMB_RATE) | (region_final["climb_rate_fpm"].isna())
].copy()
print("region_final after corrected filter:", region_final.shape)

NameError: name 'region_final' is not defined

In [1]:
python scripts/07_phase_features.py

SyntaxError: invalid decimal literal (1403692550.py, line 1)